# NS-SDN (Non-Stationary Spectral Decomposition Network) For Econometric Forecasting

Nikhil Sunder \
Student | ICSRI Research Fellow | Open-Source Developer \
B.S.B.A. Quantitative Economics, Finance & Minor in Math \
University of Miami Herbert Business School \
[Github](https://github.com/nikhilxsunder) | [PyPI](https://pypi.org/user/nikhil.sunder/) | [Anaconda](https://anaconda.org/nikhil.sunder/) | [LinkedIn](https://www.linkedin.com/in/nikhil-sunder/) \
[ORCID](https://orcid.org/0009-0007-3323-1760) | [Zenodo](https://zenodo.org/search?q=metadata.creators.person_or_org.name:%22Sunder,+Nikhil%22) | [Handshake](https://miami.joinhandshake.com/profiles/6pcqp4) \
nss106@miami.edu \
(305) 409-3108

## Introduction

Recent work in neural implicit representations has shown that sinusoidal nonlinearities can represent highly oscillatory structure with strong gradient flow properties. In particular, Sinusoidal Representation Networks (SIREN) replace conventional activations (e.g., sigmoid or ${\tanh}$) with ${\sin(\cdot)}$, enabling compact representation of periodic structure. This is well-aligned with domains such as signal processing and physics-informed modeling, where harmonic structure is intrinsic.

This proposal asks whether sinusoidal function representations can be adapted to econometric time series forecasting, where observables often exhibit both secular trend and cyclical variation, and where cycles may be nonstationary (time-varying amplitude, time-varying frequency, phase shifts after shocks). The goal is a forecasting architecture that explicitly decomposes a series into interpretable components and adapts these components over time via a latent state.

## Design Concept

### Econometric decomposition target

Assume an observed scalar time series ${y(t)}$ admits an additive decomposition into trend, cyclical structure, and idiosyncratic error:
$$
{y(t)=T(t)+C(t)+\epsilon(t)}
$$
Rather than imposing fixed-frequency cycles, NS-SDN models the cyclical portion as a sum of nonstationary sinusoidal components whose amplitude and frequency evolve through time.

## Network Overview

### From stationary sinusoid to nonstationary spectral sum

A single stationary sinusoid is inadequate for macro/financial data because amplitude/frequency are rarely constant. NS-SDN therefore models the signal as:
$$
{f(t)=B(t)+\sum{k=1}{K} A_k(t)\sin(\theta_k(t)+\varphi_k(t))}
$$
Where:
* ${B(t)}$ is the trend/low-frequency component,
* ${A_k(t)}$ is the (positive) time-varying amplitude (“envelope”) of component ${k}$,
* ${\theta_k(t)}$ is the cumulative phase of component ${k}$,
* ${\varphi_k(t)}$ is a time-varying phase offset (“horizontal shift”) that allows alignment changes without forcing distortion into amplitude/frequency.

The defining feature is that ${B(t)}$, ${A_k(t)}$, ${\theta_k(t)}$, ${\varphi_k(t)}$ are functions of discrete time through a learned latent state.

## Discrete-Time State-Space NS-SDN (final architecture)

Let the time grid be ${t_0<t_1<\dots<t_N}$, with ${\Delta t_n:=t_n-t_{n-1}}$ for ${n\ge1}$. Let the observed data be ${y_n:=y(t_n)}$ and the model output be ${\hat y_n:=f(t_n)}$.

### Latent state with transition equation (state-space / RNN-style)

NS-SDN is formulated as a nonlinear state-space model:
\
\
**State transition**
$$
{
h_n = F_\psi(h_{n-1}, x_n, \Delta t_n)
}
\qquad
h_n\in\mathbb{R}^d,
$$

where ${x_n}$ is the information available at time ${t_n}$ (e.g., ${x_n=y_n}$ for univariate autoregressive forecasting, or a vector of predictors in the multivariate case). This transition can be instantiated as a GRU/LSTM cell, or as a time-aware update of the form ${h_n = h_{n-1} + \Delta t_n \cdot g_\psi(h_{n-1}, x_n)}$, which is a discrete-time analog of continuous-time dynamics.

**Emission (spectral synthesis)**

Given ${h_n}$, the model emits ${\hat y_n}$ via nonstationary spectral synthesis.

### Emission components (trend, amplitude, frequency, phase, offset)

**Trend component**

${B_n:=B(t_n)=w_B^\top h_n+b_B}$.

**Amplitude (positive envelope)**

${A_n=\text{softplus}(W_A h_n+b_A)\in\mathbb{R}_{>0}^{K},
\qquad
A_{n,k}=\text{softplus}(w_{A,k}^\top h_n+b_{A,k}).}$

**Instantaneous angular frequency**

${\omega_n=\omega_{\text{base}}+W_\omega h_n+b_\omega\in\mathbb{R}^K,
\qquad
\omega_{n,k}=\omega_{\text{base},k}+w_{\omega,k}^\top h_n+b_{\omega,k}}$.

(Design note: ${\omega_n}$ may be left unconstrained in ${\mathbb{R}^K}$, or constrained positive via softplus if interpretability/identifiability is desired.)

**Cumulative phase (discrete-time integration)**

${\theta_{0,k}\ \text{learned},\qquad
\theta_{n,k}=\theta_{n-1,k}+\omega_{n,k}\Delta t_n\quad (n\ge1).}$

**Phase offset (horizontal shift from state)**

${\varphi_n=W_\varphi h_n+b_\varphi\in\mathbb{R}^K,
\qquad
\varphi_{n,k}=w_{\varphi,k}^\top h_n+b_{\varphi,k}.}$

### Final NS-SDN output equation

The emission is:

$$
{
\hat y_n
=
B_n
+
\sum_{k=1}^{K} A_{n,k}\,\sin(\theta_{n,k}+\varphi_{n,k})
}
$$

Optionally, for numerical symmetry one may use the trigonometric identity:

$$
{\sin(\theta+\varphi)=\sin(\theta)\cos(\varphi)+\cos(\theta)\sin(\varphi),}
$$

so:

$$
{\hat y_n
=
B_n
+
\sum_{k=1}^{K} A_{n,k}\Big[
\sin(\theta_{n,k})\cos(\varphi_{n,k})
+
\cos(\theta_{n,k})\sin(\varphi_{n,k})
\Big].}
$$

## Interpretation and positioning

The resulting architecture is closely related to FM synthesis / neural oscillator models, but differs in two critical ways:

1. **Nonstationary envelopes and frequencies are state-driven:**
${A_n}$ and ${\omega_n}$ evolve with ${h_n}$, allowing time-varying cycles rather than fixed Fourier bases.

2. **Econometric state-space structure:**
The model explicitly separates latent dynamics ${h_n}$ (transition equation) from measurement construction ${\hat y_n}$ (spectral emission), paralleling classical trend-cycle state-space decomposition while allowing nonlinear, data-driven evolution.

For these reasons, the architecture is naturally described as a Non-Stationary Spectral Decomposition Network (NS-SDN).

## Econometric Application

### Targeted use case

The proposed NS-SDN architecture is designed for macroeconomic and financial time series that exhibit both secular trend and cyclical structure, where the amplitude, frequency, and phase of cycles may evolve over time due to regime changes, policy shocks, or structural breaks.

Canonical examples include:
* real GDP and output gap measures
* inflation and inflation expectations
* interest rates and yield spreads
* unemployment and labor market indicators
* commodity prices and real exchange rates

These series are known to exhibit:
1. persistent low-frequency trends,
2. medium-frequency business-cycle dynamics,
3. episodic nonstationarity following shocks (e.g. monetary tightening, supply disruptions).

Traditional linear models struggle to jointly capture these features without strong parametric assumptions.

### Mapping NS-SDN to econometric structure

NS-SDN can be interpreted as a nonlinear, nonstationary unobserved-components model, where:
* the latent state ${h_n}$ plays the role of an unobserved economic state,
* the trend component ${B_n}$ corresponds to a stochastic trend,
* the sinusoidal components correspond to stochastic cycles with time-varying amplitude and frequency,
* the observation equation is explicitly additive and interpretable.

Unlike classical unobserved components models or Kalman-filter-based stochastic cycle models, NS-SDN:
* does not impose fixed frequencies,
* does not assume linear Gaussian dynamics,
* allows regime-dependent evolution of spectral structure.

This makes NS-SDN particularly suitable for environments where cyclical behavior itself is nonstationary, such as post-crisis macroeconomic regimes.

### Forecasting task formulation

Let ${\{y_1,\dots,y_T\}}$ denote an observed time series. The forecasting task is defined as predicting future values:

${\hat y_{T+1}}$, ${\hat y_{T+2}}$, ${\dots}$, ${\hat y_{T+H}}$

given information available up to time ${T}$.

NS-SDN is trained in a recursive one-step-ahead forecasting framework, where:
* the latent state ${h_n}$ is updated sequentially,
* predictions are generated via the spectral emission equation,
* multi-step forecasts are obtained by iterating the state transition forward without observing new data.

This setup mirrors the forecasting protocols used for VARs, state-space models, and neural sequence models, enabling fair comparison.

## Evaluation and Forecasting Efficacy

### Benchmark models

To evaluate the forecasting performance of NS-SDN, it will be compared against a set of widely used econometric and machine learning benchmarks, chosen to span increasing levels of model flexibility:

**Linear econometric models**
* AR(p)
* ARIMA(p,d,q)
* VAR (for multivariate extensions)

**State-space and decomposition models**
* local-level / local-trend models
* unobserved components models with stochastic cycles
* Kalman-filter-based trend–cycle decompositions

**Neural network baselines**
* feedforward neural networks (MLP)
* recurrent neural networks (LSTM / GRU)

These benchmarks are well understood, commonly used in applied econometrics, and provide a meaningful reference for assessing gains from nonstationary spectral modeling.

### Training and evaluation protocol

To ensure comparability across models, all methods will be evaluated under a rolling-origin out-of-sample forecasting design:
1. Split the data into an initial estimation window and an evaluation window.
2. Estimate each model using data up to time t.
3. Generate forecasts for horizons h = 1, 4, 8, 12 (depending on data frequency).
4. Roll the estimation window forward and repeat.

This procedure mimics real-time forecasting and avoids look-ahead bias.

### Forecast accuracy metrics

Forecast performance will be assessed using standard loss functions:
* Mean Squared Error (MSE)
* Root Mean Squared Error (RMSE)
* Mean Absolute Error (MAE)

For each horizon *${h}$*, loss is computed as:
${\text{MSE}(h)=\frac{1}{N}\sum_{i=1}^N (y_{t_i+h}-\hat y_{t_i+h})^2}$.

Results will be reported both per horizon and aggregated across horizons, enabling comparison of short-term versus medium-term forecasting ability.


### Statistical comparison of forecasting performance

To formally assess whether NS-SDN delivers statistically significant improvements, forecast errors will be compared using standard forecast comparison tests, such as:
* Diebold–Mariano tests for equal predictive accuracy
* horizon-specific loss differentials
* cumulative squared forecast error plots

This ensures that any observed improvements are not due to sampling variability alone.

### Diagnostic analysis and interpretability

Beyond point forecast accuracy, NS-SDN allows inspection of learned internal components:
* time-varying amplitudes ${A_{n,k}}$,
* evolving frequencies ${\omega_{n,k}}$,
* inferred latent state trajectories ${h_n}$.

These diagnostics will be analyzed to determine whether the model captures economically meaningful patterns, such as:
* changes in business-cycle duration,
* post-shock damping or amplification,
* regime-dependent cyclical behavior.

This interpretability distinguishes NS-SDN from black-box neural forecasting models.

### Hypotheses

The empirical analysis will test the following hypotheses:
1. **Forecasting hypothesis:**
NS-SDN achieves lower out-of-sample forecast error than traditional linear econometric models for medium-horizon forecasts.
2. **Nonstationarity hypothesis:**
Gains from NS-SDN are largest during periods of structural change or regime transition.
3. **Interpretability hypothesis:**
Learned spectral components correspond to economically plausible cyclical dynamics rather than noise-fitting.